# Subconjuntos de características suficientes

In [2]:
import warnings, numpy as np, pandas as pd
from pathlib import Path
from itertools import combinations
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold
warnings.filterwarnings("ignore")
SEED = 42
UMBRAL_LEAKAGE = 0.9
DEPTH = 4

NSL = (Path.home() / ".cache" / "kagglehub" / "datasets" / "hassan06" / "nslkdd" / "versions" / "1" / "KDDTrain+_20Percent.txt")
DS2 = (Path.home() / ".cache" / "kagglehub" / "datasets" / "libamariyam" / "ds2os-dataset" / "versions" / "1" / "DS2OS.csv")
MIR = (Path.cwd().parent / "data" / "flows.csv")


In [3]:
# Carga + preprocesado canonico (binario ataque/normal)
def load_nsl():
    cols=["duration","protocol_type","service","flag","src_bytes","dst_bytes","land","wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised","root_shell","su_attempted","num_root","num_file_creations","num_shells","num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count","srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count","dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate","dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate","label","difficulty"]
    df=pd.read_csv(NSL,header=None,names=cols).drop(columns=["difficulty"])
    feats=["src_bytes","service","dst_bytes","flag","diff_srv_rate","same_srv_rate","dst_host_srv_count","dst_host_same_srv_rate"]
    for c in ["service","flag"]: df[c]=LabelEncoder().fit_transform(df[c].astype(str))
    y=(df["label"].str.lower()!="normal").astype(int).values
    return df[feats].astype(float),y,feats

def load_ds2os():
    df=pd.read_csv(DS2); df["accessedNodeType"]=df["accessedNodeType"].fillna("Malicious")
    df["value"]=pd.to_numeric(df["value"].replace({"False":0,"True":1,"Twenty":20,"none":0}),errors="coerce").fillna(0)
    CAT=["sourceID","sourceAddress","sourceType","sourceLocation","destinationServiceAddress","destinationServiceType","destinationLocation","accessedNodeAddress","accessedNodeType","operation"]
    for c in CAT: df[c]=LabelEncoder().fit_transform(df[c].astype(str))
    feats=CAT+["value"]; y=(df["normality"].str.lower()!="normal").astype(int).values
    return df[feats].astype(float),y,feats

def load_mirai():
    df=pd.read_csv(MIR); df["src_ip"]=df["src_ip"].astype(str)
    def rc(r):
        p,b,a=r["state"],r["b_pkts"],r["avg_pkt_size"]
        if p=="DNS":return "DNS_FLOOD"
        if p in("HTTP","HTTPS"):return "HTTP_FLOOD"
        if p=="SSH":return "OTHER"
        if p=="UDP_OTHER":return "UDP_SMALL_NORESPONSE" if (b==0 and a<100) else ("UDP_LARGE_NORESPONSE" if b==0 else "UDP_BIDIRECTIONAL")
        if p=="TCP_OTHER":return "TCP_SYN_LIKE" if (b==0 and a<80) else ("TCP_ACK_LIKE" if b==0 else "TCP_ESTABLISHED")
        return "OTHER"
    df["state"]=df.apply(rc,axis=1)
    HW={"Normal":68200+8525,"ACK_Flood":6600+825,"DNS_Flood":4312+539,"Mirai_CnC":68200+8525,"GREIP_Flood":24712+3089,"HTTP_Flood":120+15,"SYN_Flood":68200+8525,"UDP_Flood":28816+3062,"VSE_Flood":4432+554}
    np.random.seed(42); keep=[]
    for cn,n in HW.items():
        idx=np.where(df["class_name"].values==cn)[0]
        if len(idx)==0: continue
        if len(idx)>n: idx=np.random.choice(idx,n,replace=False)
        keep.append(idx)
    df=df.iloc[np.sort(np.concatenate(keep))].reset_index(drop=True)
    feats=["n_pkts","n_bytes","f_pkts","f_bytes","b_pkts","b_bytes","avg_pkt_size","duration","state"]
    df["state"]=LabelEncoder().fit_transform(df["state"].astype(str))
    y=df["label"].astype(int).values
    return df[feats].astype(float),y,feats


In [4]:
# Funciones
def preparar(loader, sample_n=None):
    Xdf, y, feats = loader()
    X = Xdf.values
    if sample_n and len(y) > sample_n:                  # submuestreo solo para acelerar
        rng = np.random.RandomState(SEED)
        idx = np.hstack([rng.choice(np.where(y==c)[0], int(round(sample_n*(y==c).mean())), replace=False)
                         for c in np.unique(y)])
        X, y = X[idx], y[idx]
    cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
    return X, y, feats, cv

def _f1_auc(X, y, cv, cols):
    sc = cross_validate(DecisionTreeClassifier(max_depth=DEPTH, class_weight="balanced", random_state=SEED),
                        X[:, cols], y, cv=cv, scoring=["f1_macro","roc_auc"], n_jobs=-1)
    return sc["test_f1_macro"].mean(), sc["test_roc_auc"].mean()

def individuales(name, X, y, feats, cv):
    rows = [(f, *_f1_auc(X, y, cv, [i])) for i, f in enumerate(feats)]
    tab = (pd.DataFrame(rows, columns=["variable","F1_solo","AUC_solo"])
             .sort_values("F1_solo", ascending=False).reset_index(drop=True))
    tab["leakage"] = tab["F1_solo"] >= UMBRAL_LEAKAGE
    tab.to_csv(str(Path.cwd().parent / f"una_variable_{name}.csv"), index=False)
    print(f"== {name}: F1 por caracteristica individual (umbral leakage = {UMBRAL_LEAKAGE}) ==")
    print(tab.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    excluidas = tab.loc[tab["leakage"], "variable"].tolist()
    print(f"\n>> Excluidas por superar el umbral (leakage): {excluidas if excluidas else 'ninguna'}")
    return excluidas

def parejas_trios(name, X, y, feats, cv, excluir):
    keep = [i for i, f in enumerate(feats) if f not in excluir]
    f1_full, _ = _f1_auc(X, y, cv, keep)   # referencia: todas las NO excluidas juntas
    rows = []
    for r in (2, 3):
        for combo in combinations(keep, r):
            f1, auc = _f1_auc(X, y, cv, list(combo))
            rows.append((r, " + ".join(feats[i] for i in combo), f1, auc))
    res = (pd.DataFrame(rows, columns=["r","features","f1_macro","auc"])
             .sort_values("f1_macro", ascending=False).reset_index(drop=True))
    res.to_csv(str(Path.cwd().parent / f"subsets_{name}.csv"), index=False)
    fmt = lambda v: f"{v:.4f}"
    print(f"== {name}: parejas/trios SIN las features excluidas ==")
    print(f"(referencia: todas las features no excluidas juntas -> F1 {f1_full:.4f})\n")
    print("TOP 10:")
    print(res.head(10).to_string(index=False, float_format=fmt))
    for r in (2, 3):
        print(f"\nMejores r={r}:")
        print(res[res.r==r].head(3).to_string(index=False, float_format=fmt))
    return res


## NSL-KDD

**1. F1 por caracteristica individual**

In [5]:
X_nslkdd, y_nslkdd, f_nslkdd, cv_nslkdd = preparar(load_nsl, sample_n=None)
excl_nslkdd = individuales("nslkdd", X_nslkdd, y_nslkdd, f_nslkdd, cv_nslkdd)


== nslkdd: F1 por caracteristica individual (umbral leakage = 0.9) ==
              variable  F1_solo  AUC_solo  leakage
             src_bytes   0.9404    0.9663     True
             dst_bytes   0.9085    0.9167     True
               service   0.8842    0.9432    False
                  flag   0.8794    0.8903    False
         same_srv_rate   0.8667    0.8747    False
         diff_srv_rate   0.8663    0.8759    False
dst_host_same_srv_rate   0.8514    0.8880    False
    dst_host_srv_count   0.8486    0.9089    False

>> Excluidas por superar el umbral (leakage): ['src_bytes', 'dst_bytes']


**2. Parejas y tríos (excluyendo las que superan el umbral)**

In [6]:
res_nslkdd = parejas_trios("nslkdd", X_nslkdd, y_nslkdd, f_nslkdd, cv_nslkdd, excl_nslkdd)


== nslkdd: parejas/trios SIN las features excluidas ==
(referencia: todas las features no excluidas juntas -> F1 0.9532)

TOP 10:
 r                                         features  f1_macro    auc
 2                                   service + flag    0.9603 0.9821
 3                   service + flag + diff_srv_rate    0.9570 0.9728
 3          service + flag + dst_host_same_srv_rate    0.9547 0.9813
 3                   service + flag + same_srv_rate    0.9523 0.9731
 3              service + flag + dst_host_srv_count    0.9503 0.9823
 3 service + same_srv_rate + dst_host_same_srv_rate    0.9379 0.9635
 3 service + diff_srv_rate + dst_host_same_srv_rate    0.9371 0.9694
 3          service + diff_srv_rate + same_srv_rate    0.9369 0.9734
 2                          service + same_srv_rate    0.9341 0.9699
 2                          service + diff_srv_rate    0.9315 0.9757

Mejores r=2:
 r                features  f1_macro    auc
 2          service + flag    0.9603 0.9821
 2 servic

## Mirai

**1. F1 por caracteristica individual**

In [7]:
X_mirai, y_mirai, f_mirai, cv_mirai = preparar(load_mirai, sample_n=None)
excl_mirai = individuales("mirai", X_mirai, y_mirai, f_mirai, cv_mirai)


== mirai: F1 por caracteristica individual (umbral leakage = 0.9) ==
    variable  F1_solo  AUC_solo  leakage
avg_pkt_size   0.9731    0.9969     True
     f_bytes   0.8264    0.9832    False
     n_bytes   0.8134    0.9824    False
    duration   0.7724    0.8249    False
      f_pkts   0.6711    0.7877    False
      n_pkts   0.6512    0.8004    False
       state   0.5791    0.8897    False
     b_bytes   0.2947    0.6495    False
      b_pkts   0.2944    0.6383    False

>> Excluidas por superar el umbral (leakage): ['avg_pkt_size']


**2. Parejas y tríos (excluyendo las que superan el umbral)**

In [8]:
res_mirai = parejas_trios("mirai", X_mirai, y_mirai, f_mirai, cv_mirai, excl_mirai)


== mirai: parejas/trios SIN las features excluidas ==
(referencia: todas las features no excluidas juntas -> F1 0.9782)

TOP 10:
 r                   features  f1_macro    auc
 3  f_bytes + b_bytes + state    0.9782 0.9983
 3   n_bytes + b_pkts + state    0.9782 0.9984
 3  n_bytes + b_bytes + state    0.9781 0.9982
 3   f_bytes + b_pkts + state    0.9781 0.9983
 3 n_bytes + duration + state    0.9734 0.9987
 3   n_bytes + f_pkts + state    0.9730 0.9988
 3   n_pkts + n_bytes + state    0.9730 0.9988
 3  n_bytes + f_bytes + state    0.9730 0.9985
 2            n_bytes + state    0.9729 0.9985
 3 f_bytes + duration + state    0.9678 0.9986

Mejores r=2:
 r         features  f1_macro    auc
 2  n_bytes + state    0.9729 0.9985
 2  f_bytes + state    0.9672 0.9983
 2 f_pkts + f_bytes    0.9508 0.9955

Mejores r=3:
 r                  features  f1_macro    auc
 3 f_bytes + b_bytes + state    0.9782 0.9983
 3  n_bytes + b_pkts + state    0.9782 0.9984
 3 n_bytes + b_bytes + state    0.9781 0

## DS2OS

**1. F1 por caracteristica individual**

In [9]:
X_ds2os, y_ds2os, f_ds2os, cv_ds2os = preparar(load_ds2os, sample_n=80000)
excl_ds2os = individuales("ds2os", X_ds2os, y_ds2os, f_ds2os, cv_ds2os)


== ds2os: F1 por caracteristica individual (umbral leakage = 0.9) ==
                 variable  F1_solo  AUC_solo  leakage
               sourceType   0.7705    0.8077    False
      destinationLocation   0.5813    0.8035    False
                 sourceID   0.5271    0.9271    False
                    value   0.5131    0.8725    False
      accessedNodeAddress   0.5107    0.8283    False
destinationServiceAddress   0.5106    0.8277    False
   destinationServiceType   0.4826    0.7985    False
           sourceLocation   0.4742    0.8204    False
         accessedNodeType   0.4308    0.7282    False
            sourceAddress   0.3896    0.8333    False
                operation   0.2660    0.5650    False

>> Excluidas por superar el umbral (leakage): ninguna


**2. Parejas y tríos (excluyendo las que superan el umbral)**

In [10]:
res_ds2os = parejas_trios("ds2os", X_ds2os, y_ds2os, f_ds2os, cv_ds2os, excl_ds2os)


== ds2os: parejas/trios SIN las features excluidas ==
(referencia: todas las features no excluidas juntas -> F1 0.8891)

TOP 10:
 r                                                      features  f1_macro    auc
 3                     sourceID + destinationServiceType + value    0.8911 0.9859
 3                 sourceID + destinationServiceType + operation    0.8645 0.9672
 3                sourceID + sourceType + destinationServiceType    0.8644 0.9668
 3             sourceID + sourceAddress + destinationServiceType    0.8640 0.9668
 2                             sourceID + destinationServiceType    0.8640 0.9668
 3       sourceID + destinationServiceType + destinationLocation    0.8640 0.9668
 3          sourceID + destinationServiceType + accessedNodeType    0.8634 0.9554
 3       sourceID + destinationServiceType + accessedNodeAddress    0.8619 0.9616
 3 sourceID + destinationServiceAddress + destinationServiceType    0.8619 0.9616
 3                      sourceID + sourceType + acc